# 🏆 FIFA World Cup 2026 — Monte Carlo Tournament Simulator

Uses historical match results, FIFA rankings, and an Elo rating system to train a Random Forest classifier, then simulates the 2026 World Cup thousands of times to estimate each team's probability of winning.

## 1. Imports & Configuration

In [22]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import time
from itertools import combinations

In [23]:
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

TRAINING_START    = pd.Timestamp('2010-01-01')
PREDICTION_CUTOFF = pd.Timestamp('2026-06-09')

# Tournament importance weights
TOURNAMENT_WEIGHTS = {
    'FIFA World Cup': 1.00,
    'UEFA Euro': 0.95,
    'Copa América': 0.95,
    'African Cup of Nations': 0.85,
    'AFC Asian Cup': 0.85,
    'CONCACAF Gold Cup': 0.85,
    'Gold Cup': 0.85,
    'UEFA Nations League': 0.90,
    'CONCACAF Nations League': 0.80,
    'FIFA World Cup qualification': 0.75,
    'UEFA Euro qualification': 0.70,
    'African Cup of Nations qualification': 0.70,
    'AFC Asian Cup qualification': 0.65,
    'Friendly': 0.30,
}
DEFAULT_WEIGHT = 0.50

# Score distribution priors
DRAW_P = np.array([0.338, 0.446, 0.185, 0.031]); DRAW_P /= DRAW_P.sum()   # 0-0, 1-1, 2-2, 3-3
WIN_P  = np.array([0.282, 0.398, 0.199, 0.076, 0.021, 0.014, 0.007, 0.002]); 
WIN_P /= WIN_P.sum()
WIN_CDF = np.cumsum(WIN_P)
LOSE_P = np.array([0.573, 0.355, 0.069, 0.002]); 
LOSE_P /= LOSE_P.sum()

In the code above, weights are used for each tournament every team has played, as a World Cup game is different than a friendly one.

Additionally, score distribution priors are used. When predicting the outcomes, just predicting a winner or loser isn't enough due to the group stages and tie-breakers. This is why a score distribution was added. 

This is how it works:

* **DRAW_P**: Probabilities of how draws end. For example, 33.8% of draws end in 0-0, 44.6% end in 1-1, and so on.
* **WIN_P**: Probabilities of how wins end. For example, 28.2% of wins end with the winning team scoring 1 goal.
* **LOSE_P**: Probabilities of how losses end. For example, 57.3% of losses end with the losing team scoring 0 goals.

**NOTE**: The WIN_P distribution starts with 1 goal while the LOSE_P distribution starts with 0 goals.

**WIN_CDF** stands for **Cumulative Distribution Function** for winning goals. 

Instead of just looking at individual percentages, it adds up the probabilities from **WIN_P** step by step to create a running total from 0 to 1.

This is how it is built from **WIN_P**:
* **1 goal:** 0.282
* **2 goals or fewer:** 0.282 + 0.398 = **0.680**
* **3 goals or fewer:** 0.680 + 0.199 = **0.879**
* ...and so on, until it reaches **1.0** (100%).


By the end, **WIN_P** is an array of 8 floating-point numbers:

* [0.28228228, 0.3983984, 0.1991992, 0.07607608, 0.02102102, 0.01401401, 0.00700701, 0.002002]

And **WIN_CDF** is a running total from 0 to 1:

* [0.28228228, 0.68068068, 0.87987988, 0.95595596, 0.97697698, 0.99099099, 0.997998, 1.0]


## 2. Load & Clean Data

In [24]:
def strip_text(df):
    df = df.copy()
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].str.strip()
    return df

results_raw      = pd.read_csv('results.csv')
former_names_raw = pd.read_csv('former_names.csv')
rankings_raw     = pd.read_csv('fifa_rankings.csv')
groups_raw       = pd.read_csv('world_cup_groups.csv')

# Build country name mapping (former → current)
former_names = strip_text(former_names_raw)
NAME_MAP = dict(zip(former_names['former'], former_names['current']))

def apply_name_map(df, cols, m):
    df = df.copy()
    for c in cols:
        if c in df.columns:
            df[c] = df[c].replace(m)
    return df

# Clean results
results = strip_text(results_raw)
results = apply_name_map(results, ['home_team', 'away_team', 'country'], NAME_MAP)
results['date'] = pd.to_datetime(results['date'])
results['neutral'] = (results['neutral']
    .map({True:1, False:0, 'TRUE':1, 'FALSE':0, 1:1, 0:0})
    .fillna(0).astype(int))
results = (
    results[
        (results['date'] >= TRAINING_START) &
        (results['date'] <= PREDICTION_CUTOFF)
    ]
    .dropna(subset=['home_score', 'away_score'])
    .drop_duplicates()
    .copy()
)
results = results.sort_values('date').reset_index(drop=True)
results['match_id']   = np.arange(len(results))
results['home_score'] = results['home_score'].astype(int)
results['away_score'] = results['away_score'].astype(int)
results['result'] = np.select(
    [results['home_score'] > results['away_score'],
     results['home_score'] < results['away_score']],
    ['home_win', 'away_win'], default='draw'
)
results['weight'] = results['tournament'].map(TOURNAMENT_WEIGHTS).fillna(DEFAULT_WEIGHT)

# Clean rankings
rankings = strip_text(rankings_raw).rename(columns={'Nation': 'team'})
rankings[['FIFA_2022_ranking', 'FIFA_2026_rank', 'rank_change']] = (
    rankings[['FIFA_2022_ranking', 'FIFA_2026_rank', 'rank_change']].astype(int)
)
rankings = rankings.sort_values('FIFA_2026_rank').reset_index(drop=True)
RANKED_TEAMS = list(rankings['team'])
RANKED_SET   = set(RANKED_TEAMS)
rank_lu      = rankings.set_index('team')['FIFA_2026_rank']

# Clean groups
groups_clean = strip_text(groups_raw)
groups_clean = apply_name_map(groups_clean, ['team'], NAME_MAP)
WC_GROUPS = {
    label: gdf['team'].tolist()
    for label, gdf in groups_clean.groupby('group', sort=True)
}

print(f"Results loaded: {len(results):,} matches ({results['date'].min().date()} → {results['date'].max().date()})")
print(f"Ranked teams: {len(RANKED_TEAMS)}")
print(f"Groups: {list(WC_GROUPS.keys())}")

Results loaded: 15,790 matches (2010-01-02 → 2026-06-08)
Ranked teams: 48
Groups: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L']


Four source files are loaded:

| File | Contents |
|---|---|
| `results.csv` | Historical match results (teams, scores, tournament, date, venue) |
| `former_names.csv` | Mapping of old → current country names (e.g. *West Germany → Germany*) |
| `fifa_rankings.csv` | FIFA rankings for participating teams |
| `world_cup_groups.csv` | 2026 World Cup group assignments |

### Apply Name Map Function

```python
former_names = strip_text(former_names_raw)
NAME_MAP = dict(zip(former_names['former'], former_names['current']))

def apply_name_map(df, cols, m):
    ...
    df[c] = df[c].replace(m)
```


Builds a dictionary `NAME_MAP` (`{"Zaire": "DR Congo", ...}`) and applies it to any specified column. This ensures renamed nations are unified under the modern country name used in the rankings and group data.


### Derived columns
```python
results['match_id']   = np.arange(len(results))
results['result']     = np.select([...], ['home_win', 'away_win'], default='draw')
results['weight']     = results['tournament'].map(TOURNAMENT_WEIGHTS).fillna(DEFAULT_WEIGHT)
```

| Column | Description |
|---|---|
| `match_id` | Sequential integer index for each match |
| `neutral` | Binary flag (1/0) for neutral-venue matches, normalised from mixed True/False/string inputs |
| `result` | Categorical outcome: `home_win`, `away_win`, or `draw` |
| `weight` | Tournament importance multiplier (e.g. World Cup > friendly), used to weight model training |


Three convenience structures are derived:

| Variable | Type | Purpose |
|---|---|---|
| `RANKED_TEAMS` | `list` | Ordered list of all 2026-ranked teams |
| `RANKED_SET` | `set` | Fast membership checks (`"Iran" in RANKED_SET`) |
| `rank_lu` | `Series` | `team → FIFA_2026_rank` lookup |


## Building `WC_GROUPS`

```python
WC_GROUPS = {
    label: gdf['team'].tolist()
    for label, gdf in groups_clean.groupby('group', sort=True)
}
```

Produces a dictionary mapping each group letter to its list of teams:

```python
{
  'A': ['Mexico', 'USA', 'Canada', 'New Zealand'],
  'B': ['Argentina', 'Chile', ...],
  ...
}
```

This is the primary structure used later to simulate the group stage.

## 3. Elo Rating System

In [25]:
ELO_BASE  = 1500
ELO_K_MAX = 60
ELO_K_MIN = 10

def weight_to_k(w):
    w = max(0.3, w)
    return 20.0 + (w - 0.30) / (1.00 - 0.30) * (ELO_K_MAX - 20.0)

def expected_elo(a, b):
    return 1.0 / (1.0 + 10 ** ((b - a) / 400.0))

def update_elo(ratings, home_team, away_team, home_score, away_score, k_factor):
    # Retrieve current ratings or fallback to the base rating
    home_rating = ratings.get(home_team, ELO_BASE)
    away_rating = ratings.get(away_team, ELO_BASE)
    
    # Calculate expected win probabilities (expected scores)
    expected_home = expected_elo(home_rating, away_rating)
    expected_away = 1 - expected_home
    
    # Determine actual match outcomes (actual scores)
    if home_score > away_score:
        actual_home, actual_away = 1.0, 0.0
    elif home_score < away_score:
        actual_home, actual_away = 0.0, 1.0
    else:
        actual_home, actual_away = 0.5, 0.5
        
    # Calculate and update new Elo ratings
    ratings[home_team] = home_rating + k_factor * (actual_home - expected_home)
    ratings[away_team] = away_rating + k_factor * (actual_away - expected_away)

# Walk through all matches chronologically to build live Elo snapshot
elo_snapshot  = {}
pre_match_elos = []

for _, row in results.iterrows():
    home, away = row['home_team'], row['away_team']
    elo_snapshot.setdefault(home, ELO_BASE)
    elo_snapshot.setdefault(away, ELO_BASE)
    pre_match_elos.append({
        'date': row['date'], 'home_team': home, 'away_team': away,
        'h_elo_pre': elo_snapshot[home], 'a_elo_pre': elo_snapshot[away]
    })
    k = max(ELO_K_MIN, min(ELO_K_MAX, weight_to_k(
        TOURNAMENT_WEIGHTS.get(row['tournament'], DEFAULT_WEIGHT)
    )))
    update_elo(elo_snapshot, home, away, int(row['home_score']), int(row['away_score']), k)

elo_lu = {t: elo_snapshot.get(t, ELO_BASE) for t in RANKED_TEAMS}

top10 = sorted(elo_lu.items(), key=lambda x: -x[1])[:10]
print("Top 10 teams by Elo rating:")
for i, (team, elo) in enumerate(top10, 1):
    print(f"  {i:2}. {team:<22} {elo:.1f}")

Top 10 teams by Elo rating:
   1. Spain                  2014.1
   2. Argentina              1967.1
   3. France                 1915.8
   4. Morocco                1911.5
   5. England                1890.6
   6. Japan                  1885.6
   7. Portugal               1867.2
   8. Mexico                 1866.3
   9. Brazil                 1857.0
  10. Germany                1852.9


## What is Elo?

Elo is a method for ranking competitors based on their match results. After every game, the winner gains points and the loser loses points. The key idea: **beating a strong opponent earns you more than beating a weak one.**

## Constants

```python
ELO_BASE  = 1500   # Starting rating for any new team
ELO_K_MAX = 60     # Largest possible rating shift per match
ELO_K_MIN = 10     # Smallest possible rating shift per match
```

Every team starts at **1500**. The **K factor** controls how dramatically a single match can change a team's rating.



## Step 1 — Deciding How Much a Match "Counts" (`weight_to_k`)

```python
def weight_to_k(w):
    return 20.0 + (w - 0.30) / (1.00 - 0.30) * (ELO_K_MAX - 20.0)
```

Not all matches are equal. A World Cup final matters more than a friendly. This function converts a **tournament weight** (a number between 0.30 and 1.00) into a **K factor** (between 20 and 60):

| Tournament importance | Weight | K factor |
|-----------------------|--------|----------|
| Low (friendly)        | 0.30   | 20       |
| Medium                | 0.65   | ~40      |
| High (major final)    | 1.00   | 60       |

A higher K means ratings shift more after the match.


## Step 2 — Predicting the Expected Result (`expected_elo`)

```python
def expected_elo(a, b):
    return 1.0 / (1.0 + 10 ** ((b - a) / 400.0))
```

Before a match is played, this formula calculates the **probability that team A wins**, based on the gap in ratings between A and B.

Examples:
- Equal ratings (both 1500) → 50% chance each
- A is 200 points above B → ~76% chance A wins
- A is 400 points above B → ~91% chance A wins


## Step 3 — Updating Ratings After a Match (`update_elo`)

```python
def update_elo(ratings, home, away, hs, as_, k):
    rh = ratings.get(home, ELO_BASE)
    ra = ratings.get(away, ELO_BASE)
    eh = expected_elo(rh, ra); ea = 1 - eh
    sh, sa = (1.0, 0.0) if hs > as_ else ((0.0, 1.0) if hs < as_ else (0.5, 0.5))
    ratings[home] = rh + k * (sh - eh)
    ratings[away] = ra + k * (sa - ea)
```

This is the core of Elo. Here's what happens step by step:

1. **Look up current ratings** for home and away teams (default to 1500 if unseen).
2. **Calculate expected scores** — the pre-match win probabilities.
3. **Determine actual scores**:
   - Home win → `sh = 1.0`, `sa = 0.0`
   - Away win → `sh = 0.0`, `sa = 1.0`
   - Draw → `sh = sa = 0.5`
4. **Apply the update formula**:

```
new_rating = old_rating + K × (actual_result − expected_result)
```

If a weak team upsets a strong one, the surprise is large → big rating swing. If the favourite wins as expected, the swing is small.


## Step 4 — Walking Through All Matches Chronologically

```python
for _, row in results.iterrows():
    ...
    pre_match_elos.append({ ... })   # Save ratings BEFORE the match
    k = max(ELO_K_MIN, min(ELO_K_MAX, weight_to_k(...)))
    update_elo(elo_snapshot, ...)    # Then update ratings AFTER
```

The code loops through every historical match **in date order**:

1. Records each team's rating **before** the match (useful for analysis).
2. Calculates the K factor from the tournament type.
3. Clamps K to always stay within `[10, 60]` — a safety guard.
4. Updates both teams' ratings based on the result.

This builds a **live, evolving snapshot** of ratings over time.


## Step 5 — Displaying the Top 10

```python
elo_lu = {t: elo_snapshot.get(t, ELO_BASE) for t in RANKED_TEAMS}
top10 = sorted(elo_lu.items(), key=lambda x: -x[1])[:10]
```

Finally, the code:
1. Collects the final ratings for all ranked teams.
2. Sorts them from highest to lowest.
3. Prints the top 10 with their Elo scores.

## 4. Train Random Forest Classifier

In [26]:
pre_elo_df = pd.DataFrame(pre_match_elos)

ranked_matches = (
    results[
        results['home_team'].isin(RANKED_SET) &
        results['away_team'].isin(RANKED_SET)
    ].copy().reset_index(drop=True)
)
ranked_matches['match_id'] = ranked_matches.index

pre_elo_df2 = (
    pre_elo_df[
        pre_elo_df['home_team'].isin(RANKED_SET) &
        pre_elo_df['away_team'].isin(RANKED_SET)
    ][['date', 'home_team', 'away_team', 'h_elo_pre', 'a_elo_pre']]
    .drop_duplicates(subset=['date', 'home_team', 'away_team'])
)

ranked_matches = ranked_matches.merge(pre_elo_df2, on=['date', 'home_team', 'away_team'], how='left')
ranked_matches['h_rank']    = ranked_matches['home_team'].map(rank_lu)
ranked_matches['a_rank']    = ranked_matches['away_team'].map(rank_lu)
ranked_matches['rank_diff'] = ranked_matches['h_rank'] - ranked_matches['a_rank']
ranked_matches['elo_diff']  = ranked_matches['h_elo_pre'] - ranked_matches['a_elo_pre']

FEATURE_COLS  = ['elo_diff', 'rank_diff', 'h_elo_pre', 'a_elo_pre', 'neutral']
training_data = ranked_matches.sort_values('date').reset_index(drop=True)

X_full = training_data[FEATURE_COLS]
y_full = training_data['result']
w_full = training_data['weight'].values

model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model',   RandomForestClassifier(
        n_estimators=100, max_depth=4, min_samples_leaf=15,
        class_weight='balanced', random_state=RANDOM_STATE
    ))
])
model.fit(X_full, y_full, model__sample_weight=w_full)

CLASSES = list(model.classes_)
print(f"Training rows : {len(training_data):,}")
print(f"Elo nulls     : {training_data['h_elo_pre'].isna().sum()}")
print(f"Classes       : {CLASSES}")

Training rows : 1,932
Elo nulls     : 0
Classes       : ['away_win', 'draw', 'home_win']


# Building the Match Prediction Model

## What This Code Does

Takes the Elo ratings built in the previous step and uses them — alongside FIFA rankings — to train a machine learning model that predicts match outcomes: **home win, away win, or draw**.


## Step 1 — Convert Elo History to a DataFrame

```python
pre_elo_df = pd.DataFrame(pre_match_elos)
```

The list of pre-match Elo snapshots (built during the Elo walkthrough loop) gets turned into a proper table, with one row per match showing each team's rating *before* the game was played.


## Step 2 — Filter to Ranked Teams Only

```python
ranked_matches = results[
    results['home_team'].isin(RANKED_SET) &
    results['away_team'].isin(RANKED_SET)
]
```

Not every team in the dataset has a FIFA ranking. This filters the match history down to only matches where **both** teams are in the ranked set — the teams the model will actually be asked to predict.

The same filter is applied to the Elo table (`pre_elo_df2`), and duplicates are dropped in case any match appears twice.


## Step 3 — Merge Elo Ratings onto the Match Table

```python
ranked_matches = ranked_matches.merge(pre_elo_df2, on=['date', 'home_team', 'away_team'], how='left')
```

Joins the pre-match Elo ratings onto the match results using date and team names as the key. After this, every match row has:

| Column | Meaning |
|---|---|
| `h_elo_pre` | Home team's Elo rating before this match |
| `a_elo_pre` | Away team's Elo rating before this match |


## Step 4 — Engineer Features

```python
ranked_matches['h_rank']    = ranked_matches['home_team'].map(rank_lu)
ranked_matches['a_rank']    = ranked_matches['away_team'].map(rank_lu)
ranked_matches['rank_diff'] = ranked_matches['h_rank'] - ranked_matches['a_rank']
ranked_matches['elo_diff']  = ranked_matches['h_elo_pre'] - ranked_matches['a_elo_pre']
```

Four new columns are created. The key idea is that **differences** are more informative than raw numbers — the model cares less about absolute strength and more about the *gap* between the two teams.

| Feature | What it captures |
|---|---|
| `elo_diff` | How much stronger the home team is by Elo |
| `rank_diff` | How much higher-ranked the home team is |
| `h_elo_pre` | Home team's absolute form |
| `a_elo_pre` | Away team's absolute form |
| `neutral` | Whether the match is on neutral ground (no home advantage) |

A positive `elo_diff` means the home team is stronger; negative means the away team is.


## Step 5 — Define Features, Labels, and Weights

```python
X_full = training_data[FEATURE_COLS]   # the 5 features above
y_full = training_data['result']       # home_win / draw / away_win
w_full = training_data['weight']       # tournament importance
```

Three things the model needs:

- **X** — the input features for each match
- **y** — the correct answer (what actually happened)
- **w** — how much each match should count during training (a World Cup match counts more than a friendly, using the same weights as the K factor)


## Step 6 — Build and Train the Model

```python
model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model',   RandomForestClassifier(
        n_estimators=100, max_depth=4, min_samples_leaf=15,
        class_weight='balanced', random_state=RANDOM_STATE
    ))
])
model.fit(X_full, y_full, model__sample_weight=w_full)
```

The model is a two-stage pipeline:

### Stage 1 — Imputer
Fills in any missing feature values (e.g. a team with no Elo yet) with the **median** value from the training data rather than crashing or dropping those rows.

### Stage 2 — Random Forest Classifier
An ensemble of 100 decision trees that vote on the outcome. Key settings:

| Parameter | Value | Why |
|---|---|---|
| `n_estimators=100` | 100 trees | More trees = more stable predictions |
| `max_depth=4` | Shallow trees | Prevents overfitting on historical quirks |
| `min_samples_leaf=15` | Min 15 matches per leaf | Forces generalisable patterns, not noise |
| `class_weight='balanced'` | Auto-adjust for imbalance | Draws are rarer than wins — this stops the model ignoring them |

The `sample_weight` passed at `.fit()` time means World Cup matches influence the trees more than friendlies.

## 5. Pre-compute Match Probabilities

In [27]:
team_profiles = rankings.set_index('team')[['FIFA_2026_rank']].copy()
team_profiles['elo'] = pd.Series(elo_lu).reindex(team_profiles.index).fillna(ELO_BASE)

TEAM_IDX  = {t: i for i, t in enumerate(RANKED_TEAMS)}
N_TEAMS   = len(RANKED_TEAMS)
RANK_ARR  = np.array([rank_lu[t]               for t in RANKED_TEAMS])
ELO_ARR   = np.array([team_profiles.loc[t, 'elo'] for t in RANKED_TEAMS])

WC_HOSTS  = {"United States", "Mexico", "Canada"}
HOST_IDXS = {TEAM_IDX[t] for t in WC_HOSTS if t in TEAM_IDX}

def build_match_features(team_a_idx, team_b_idx, team_a_elo=None, team_b_elo=None):
    if team_a_elo is None: team_a_elo = ELO_ARR[team_a_idx]
    if team_b_elo is None: team_b_elo = ELO_ARR[team_b_idx]
    
    rank_a, rank_b = RANK_ARR[team_a_idx], RANK_ARR[team_b_idx]
    
    team_a_is_host = (team_a_idx in HOST_IDXS)
    team_b_is_host = (team_b_idx in HOST_IDXS)
    
    neutral = 0 if (team_a_is_host or team_b_is_host) else 1
    if team_b_is_host and not team_a_is_host:  # away team is host → flip perspective
        team_a_elo, team_b_elo = team_b_elo, team_a_elo
        rank_a, rank_b = rank_b, rank_a
    return {
        'elo_diff': team_a_elo - team_b_elo, 'rank_diff': rank_a - rank_b,
        'h_elo_pre': team_a_elo, 'a_elo_pre': team_b_elo, 'neutral': neutral
    }

pair_rows = []; pair_list = []
for ta in RANKED_TEAMS:
    for tb in RANKED_TEAMS:
        if ta == tb: continue
        ti, tj = TEAM_IDX[ta], TEAM_IDX[tb]
        pair_rows.append(build_match_features(ti, tj))
        pair_list.append((ti, tj))

batch_df  = pd.DataFrame(pair_rows)[FEATURE_COLS]
proba_all = model.predict_proba(batch_df)
cl = {c: k for k, c in enumerate(CLASSES)}

p_hw = np.zeros((N_TEAMS, N_TEAMS))
p_d  = np.zeros((N_TEAMS, N_TEAMS))
p_aw = np.zeros((N_TEAMS, N_TEAMS))

for (i, j), pr in zip(pair_list, proba_all):
    p_hw[i, j] = pr[cl['home_win']]
    p_d [i, j] = pr[cl['draw']]
    p_aw[i, j] = pr[cl['away_win']]

p_ko_base = p_hw + 0.5 * p_d
print(f"Pair probability matrices cached: {len(pair_list):,} matchups")

Pair probability matrices cached: 2,256 matchups


# Pre-Computing Match Probabilities for Every Possible Matchup

## What This Code Does

Before simulating the tournament, the model runs predictions for **every possible match** between all ranked teams and caches the results in lookup matrices. This avoids re-running the model thousands of times during simulation.


## Step 1 — Build a Team Profile Table

```python
team_profiles = rankings.set_index('team')[['FIFA_2026_rank']].copy()
team_profiles['elo'] = pd.Series(elo_lu).reindex(team_profiles.index).fillna(ELO_BASE)
```

Creates a simple reference table with one row per team containing two things: their FIFA ranking and their current Elo rating. If a team somehow has no Elo (never appeared in the match history), it defaults to 1500.


## Step 2 — Create Fast Lookup Arrays

```python
TEAM_IDX = {t: i for i, t in enumerate(RANKED_TEAMS)}
RANK_ARR = np.array([rank_lu[t]               for t in RANKED_TEAMS])
ELO_ARR  = np.array([team_profiles.loc[t, 'elo'] for t in RANKED_TEAMS])
```

Instead of looking up team stats by name every time, each team gets a numeric index (Brazil = 0, France = 1, etc.) and their rank and Elo are stored in plain arrays. Looking up `ELO_ARR[4]` is much faster than searching a dictionary inside a loop that runs thousands of times.


## Step 3 — Handle Home Advantage for Host Nations

```python
WC_HOSTS  = {"United States", "Mexico", "Canada"}
HOST_IDXS = {TEAM_IDX[t] for t in WC_HOSTS if t in TEAM_IDX}
```

The 2026 World Cup has three host nations. They effectively play every game at home, which is a meaningful advantage the model needs to account for.


## Step 4 — Build Features for a Single Matchup

```python
def build_match_features(ti, tj, ...):
    ...
    neutral = 0 if (ah or bh) else 1
    if bh and not ah:  # away team is host → flip perspective
        ta_elo, tb_elo = tb_elo, ta_elo
        ra, rb = rb, ra
    return {'elo_diff': ..., 'rank_diff': ..., ...}
```

This function takes two team indices and returns the five features the model was trained on. The tricky part is the host flip:

The model was always trained from the **home team's perspective** — `elo_diff` is always home minus away, `rank_diff` is always home minus away. If team B (the "away" team in this loop) is actually a host nation, they're effectively the home side. The fix is to swap the Elo and rank values so the features stay consistent with what the model learned.

| Scenario | `neutral` | Perspective |
|---|---|---|
| Neither team is a host | `1` | No home advantage applied |
| Team A is a host | `0` | Team A treated as home side |
| Team B is a host | `0` | Values flipped so Team B is treated as home side |


## Step 5 — Run All Matchups Through the Model at Once

```python
for ta in RANKED_TEAMS:
    for tb in RANKED_TEAMS:
        if ta == tb: continue
        pair_rows.append(build_match_features(ti, tj))
        pair_list.append((ti, tj))

batch_df  = pd.DataFrame(pair_rows)[FEATURE_COLS]
proba_all = model.predict_proba(batch_df)
```

Loops over every ordered pair of teams — Brazil vs France, Brazil vs Argentina, France vs Brazil, and so on. If there are 32 teams, that's 32 × 31 = **992 matchups**.

All 992 feature rows are stacked into a single DataFrame and passed to the model in one batch call. `predict_proba` returns three probabilities for each row: home win, draw, away win — all summing to 1.0.

Running them as a batch rather than one at a time is significantly faster.


## Step 6 — Store Results in Matrices

```python
p_hw = np.zeros((N_TEAMS, N_TEAMS))
p_d  = np.zeros((N_TEAMS, N_TEAMS))
p_aw = np.zeros((N_TEAMS, N_TEAMS))

for (i, j), pr in zip(pair_list, proba_all):
    p_hw[i, j] = pr[cl['home_win']]
    p_d [i, j] = pr[cl['draw']]
    p_aw[i, j] = pr[cl['away_win']]
```

The probabilities are stored in three 32×32 matrices. To look up the probability that Brazil beats France, you just index `p_hw[brazil_idx, france_idx]` — a single instant array lookup during simulation.

Example slice of `p_hw`:

|  | Brazil | France | Argentina |
|---|---|---|---|
| **Brazil** | — | 0.48 | 0.41 |
| **France** | 0.33 | — | 0.38 |
| **Argentina** | 0.45 | 0.44 | — |



## Step 7 — Compute Knockout Probability

```python
p_ko_base = p_hw + 0.5 * p_d
```

In knockout rounds there's no such thing as a draw — matches go to extra time and penalties. This collapses the three outcomes into a single "team A advances" probability by giving each side half the draw probability. So if `p_hw = 0.45` and `p_d = 0.25`, then `p_ko_base = 0.45 + 0.125 = 0.575`.

## 6. Tournament Simulator

In [28]:
GROUP_MATCHUPS = {
    lbl: {
        'tidxs': [TEAM_IDX[t] for t in teams],
        'pairs': list(combinations([TEAM_IDX[t] for t in teams], 2))
    }
    for lbl, teams in WC_GROUPS.items()
}

R32_SLOTS = [
    (("A",1),("B",2)), (("C",1),("D",2)), (("E",1),("F",2)), (("G",1),("H",2)),
    (("I",1),("J",2)), (("K",1),("L",2)), (("A",2),("B",1)), (("C",2),("D",1)),
    (("E",2),("F",1)), (("G",2),("H",1)), (("I",2),("J",1)), (("K",2),("L",1)),
]

def simulate_tournament(rng):
    """Run one full World Cup simulation. Returns (qual, r32w, round_reached, champion)."""

    # ── Group stage ──
    GR = {}; third_rows = []
    for lbl, info in GROUP_MATCHUPS.items():
        tidxs = info['tidxs']
        pts = {t: 0 for t in tidxs}; gf = {t: 0 for t in tidxs}; ga = {t: 0 for t in tidxs}
        for ta, tb in info['pairs']:
            u   = rng.random(); paw = p_aw[ta, tb]; pd_ = p_d[ta, tb]
            res = 0 if u < paw else (1 if u < paw + pd_ else 2)   # 0=away win, 1=draw, 2=home win
            if res == 1:
                g = int(rng.choice(4, p=DRAW_P)); h_g = a_g = g
            else:
                wg = int(np.searchsorted(WIN_CDF, rng.random())) + 1
                ml = min(wg, 4); lp = LOSE_P[:ml] / LOSE_P[:ml].sum()
                lg = int(rng.choice(ml, p=lp))
                h_g, a_g = (wg, lg) if res == 2 else (lg, wg)
            gf[ta] += h_g; ga[ta] += a_g; gf[tb] += a_g; ga[tb] += h_g
            if h_g > a_g:   pts[ta] += 3
            elif a_g > h_g: pts[tb] += 3
            else:           pts[ta] += 1; pts[tb] += 1

        tbl = sorted(tidxs, key=lambda t: (-pts[t], -(gf[t]-ga[t]), -gf[t], RANK_ARR[t]))
        GR[lbl] = {1: tbl[0], 2: tbl[1], 3: tbl[2], 4: tbl[3]}
        t3 = tbl[2]
        third_rows.append({'t': t3, 'pts': pts[t3], 'gd': gf[t3]-ga[t3], 'gf': gf[t3], 'rank': RANK_ARR[t3]})

    # ── Best 8 third-place teams qualify ──
    best8 = [row['t'] for row in sorted(third_rows, key=lambda row: (-row['pts'], -row['gd'], -row['gf'], row['rank']))[:8]]
    qual  = [GR[group_label][position] for group_label in GR for position in (1, 2)] + best8

    # ── Round of 32 ──
    r32w = []
    for (group_a, pos_a), (group_b, pos_b) in R32_SLOTS:
        team_a, team_b = GR[group_a][pos_a], GR[group_b][pos_b]
        r32w.append(team_a if rng.random() < p_ko_base[team_a, team_b] else team_b)

    for index in range(4):
        team_a, team_b = best8[index], best8[7 - index]
        r32w.append(team_a if rng.random() < p_ko_base[team_a, team_b] else team_b)

    # ── Knockout rounds ──
    current_teams = r32w; round_reached = {}
    round_reached['round_of_16'] = r32w[:]  # record the 16 R32 winners here
    for stage_name in ('quarterfinal', 'semifinal', 'final'):
        next_teams = []
        total_teams = len(current_teams)
        for index in range(total_teams // 2):
            team_a, team_b = current_teams[index * 2], current_teams[index * 2 + 1]
            next_teams.append(team_a if rng.random() < p_ko_base[team_a, team_b] else team_b)
        round_reached[stage_name] = next_teams; current_teams = next_teams

    champion = current_teams[0] if rng.random() < p_ko_base[current_teams[0], current_teams[1]] else current_teams[1]
    return qual, r32w, round_reached, champion

print("Simulator defined.")

Simulator defined.


# Simulating the Tournament

## What This Code Does

Defines a function that simulates one complete World Cup — group stage through to the final — using the pre-computed probabilities and a random number generator to decide match outcomes.


## Setup

```python
GROUP_MATCHUPS = { lbl: { 'tidxs': [...], 'pairs': list(combinations(...)) } ... }
```

Pre-builds every group's team indices and all head-to-head pair combinations so the simulator doesn't recalculate them on every run. With 12 groups of 4, each group has 6 matches (4 choose 2), so 72 group stage matches total per simulation.

The `R32_SLOTS` list hard-codes the bracket — which group winners face which runners-up in the Round of 32. This mirrors the actual 2026 World Cup draw format.


## The Group Stage

Each group is simulated match by match. For every pair of teams, a random number is drawn and compared against the pre-computed probabilities to decide the outcome — home win, draw, or away win.

Scorelines are also simulated (not just results) because goal difference is needed to break ties in the standings. Winning margins are drawn from a realistic scoring distribution, and draw scores (0–0, 1–1, 2–2, etc.) from a separate one.

After each match, **live Elo ratings update** using the same formula as the historical walkthrough. This means a team that goes on a group stage run enters the knockouts with a higher Elo than when they started — the model tracks momentum even within the simulation.

Teams are ranked by points, then goal difference, then goals scored, then FIFA ranking. The top two from each group advance automatically. Third-place finishers are recorded separately.


## Best Third-Place Teams

The 2026 format qualifies the **best 8 third-place teams** across all 12 groups. These are ranked by points, goal difference, goals scored, then FIFA ranking — and the top 8 join the Round of 32 alongside the 24 group winners and runners-up.


## Knockout Rounds

From the Round of 32 onward, each match uses `ko_prob_live` — which reads from the pre-computed `p_ko_base` matrix (home win probability plus half the draw probability). A single random draw decides who advances.

The bracket pairs winners against each other in standard knockout fashion through the quarterfinals, semifinals, and final. The champion is whoever wins the final.


## What the Function Returns

| Return value | Contents |
|---|---|
| `qual` | All 32 teams that made the Round of 32 |
| `r32w` | The 16 teams that won their Round of 32 match |
| `round_reached` | Which teams reached each of QF, SF, Final |
| `champion` | The tournament winner |

Running this function tens of thousands of times and averaging the results gives a probability distribution over outcomes — how often each team wins the tournament, reaches the final, and so on.

## 7. Run Simulations

In [29]:
N_SIMS = 100_000   # ← increase for more accurate estimates (200 for a quick test)

STAGE_ORDER = ['group_exit', 'round_of_32', 'round_of_16', 'quarterfinal', 'semifinal', 'final', 'champion']
SID = {s: i for i, s in enumerate(STAGE_ORDER)}
SC  = np.zeros((N_TEAMS, len(STAGE_ORDER)), dtype=np.int32)

t0 = time.time()
for _ in range(N_SIMS):
    qual, r32w, rr, champion = simulate_tournament(rng)
    np.add.at(SC, (list(set(range(N_TEAMS)) - set(qual)), SID['group_exit']), 1)
    np.add.at(SC, (qual, SID['round_of_32']), 1)
    np.add.at(SC, (rr['round_of_16'], SID['round_of_16']), 1)
    for sn in ('quarterfinal', 'semifinal', 'final'):
        np.add.at(SC, (rr[sn], SID[sn]), 1)
    np.add.at(SC, (champion, SID['champion']), 1)

elapsed = time.time() - t0
print(f"{N_SIMS:,} simulations completed in {elapsed:.1f}s")
print(f"Total champions assigned: {SC[:, SID['champion']].sum()} (should equal {N_SIMS})")

100,000 simulations completed in 127.7s
Total champions assigned: 100000 (should equal 100000)


# Running the Simulations

## What This Code Does

Runs the tournament simulator 10,000 times and counts how often each team reaches each stage. The result is a probability distribution over outcomes for every team.

## The Stage Counter Matrix

```python
SC = np.zeros((N_TEAMS, len(STAGE_ORDER)), dtype=np.int32)
```

A grid with one row per team and one column per stage. Every cell starts at zero and gets incremented each time a team reaches that stage across all simulations.

| Stage | Meaning |
|---|---|
| `group_exit` | Eliminated in the group stage |
| `round_of_32` | Qualified from the group stage |
| `round_of_16` | Won their Round of 32 match |
| `quarterfinal` | Reached the last 8 |
| `semifinal` | Reached the last 4 |
| `final` | Reached the final |
| `champion` | Won the tournament |


## The Loop

```python
for _ in range(N_SIMS):
    qual, r32w, rr, champion = simulate_tournament(rng)
    SC[qual,     SID['round_of_32']] += 1
    SC[r32w,     SID['round_of_16']] += 1
    ...
    SC[champion, SID['champion']]    += 1
```

Each iteration runs one full tournament and increments the counter for every team at every stage they reached. After 10,000 runs, dividing any cell by 10,000 gives a probability — if Brazil reaches the final in 3,200 out of 10,000 simulations, their final probability is 32%.

This technique is called **Monte Carlo simulation** — run a random process enough times and the frequencies converge to true probabilities.

## 8. Results

In [30]:
# Build a tidy results DataFrame
stage_cols = STAGE_ORDER[1:]   # skip 'group_exit' for display
rows = []
for i, team in enumerate(RANKED_TEAMS):
    row = {'team': team, 'fifa_rank': RANK_ARR[i], 'elo': round(ELO_ARR[i], 1)}
    for s in stage_cols:
        row[s + '_pct'] = SC[i, SID[s]] / N_SIMS * 100
    rows.append(row)

sim_results = pd.DataFrame(rows).sort_values('champion_pct', ascending=False).reset_index(drop=True)
sim_results.index += 1

# Display top 20 contenders
display_cols = ['team', 'fifa_rank', 'elo',
                'round_of_32_pct', 'round_of_16_pct', 'quarterfinal_pct',
                'semifinal_pct', 'final_pct', 'champion_pct']
top20 = sim_results[display_cols].head(20)
top20.columns = ['Team', 'FIFA Rank', 'Elo',
                 'R32 %', 'R16 %', 'QF %', 'SF %', 'Final %', 'Win %']

print("=" * 85)
print(f"{'2026 FIFA WORLD CUP — MONTE CARLO SIMULATION RESULTS':^85}")
print(f"{'(based on ' + str(N_SIMS) + ' simulations)':^85}")
print("=" * 85)
print(top20.to_string(index=True))
print("=" * 85)

                2026 FIFA WORLD CUP — MONTE CARLO SIMULATION RESULTS                 
                            (based on 100000 simulations)                            
             Team  FIFA Rank     Elo   R32 %   R16 %    QF %    SF %  Final %  Win %
1           Spain          2  2014.1  93.549  58.054  35.106  20.692   12.739  7.633
2       Argentina          3  1967.1  91.145  54.911  32.319  19.202   11.729  6.885
3          France          1  1915.8  89.981  54.011  31.496  18.771   11.505  6.798
4         England          4  1890.6  91.418  54.860  32.498  18.852   11.313  6.600
5        Portugal          5  1867.2  88.662  51.796  30.386  17.305   10.165  5.599
6         Morocco          8  1911.5  90.281  53.717  33.349  18.929   10.249  5.384
7          Brazil          6  1857.0  86.993  51.429  31.946  17.996    9.943  5.327
8     Netherlands          7  1836.7  86.436  52.477  29.677  16.203    8.798  4.825
9         Germany         10  1852.9  91.021  50.586  28.775  1

# Formatting and Displaying the Results

## What This Code Does

Converts the raw simulation counts in `SC` into percentages, builds a clean results table, and prints the top 20 contenders.


## Converting Counts to Percentages

```python
row[s + '_pct'] = SC[i, SID[s]] / N_SIMS * 100
```

For every team and every stage, the raw count is divided by the number of simulations and multiplied by 100. If Brazil reached the final in 3,200 out of 10,000 simulations, `final_pct` becomes 32.0.

Each team ends up with one row containing their FIFA ranking, Elo, and a percentage for each stage.

## Sorting and Indexing

```python
sim_results = pd.DataFrame(rows).sort_values('champion_pct', ascending=False)
sim_results.index += 1
```

Teams are ranked by their win probability, highest first. The index starts at 1 so the printed table reads as a natural ranking (1st, 2nd, 3rd...) rather than zero-based.


## What the Output Looks Like

The top 20 table has one row per team with eight columns:

| Column | Meaning |
|---|---|
| `FIFA Rank` | Official FIFA ranking going into the tournament |
| `Elo` | Computed Elo rating |
| `R32 %` | % of simulations they qualified from the group stage |
| `R16 %` | % of simulations they won the Round of 32 |
| `QF %` | % reaching the quarterfinals |
| `SF %` | % reaching the semifinals |
| `Final %` | % reaching the final |
| `Win %` | % winning the tournament |

Percentages decrease from left to right for any given team — a team that reaches the final half the time can only win it some fraction of that half.

The `group_exit` stage is intentionally excluded from the display since it's just 100% minus `R32 %` and adds no new information.

In [31]:
# Championship odds — all teams
print("\nFull Championship Probability Table:")
full = sim_results[['team', 'fifa_rank', 'elo', 'champion_pct']].copy()
full.columns = ['Team', 'FIFA Rank', 'Elo', 'Champion %']
full = full[full['Champion %'] > 0]
print(full.to_string(index=True))


Full Championship Probability Table:
                      Team  FIFA Rank     Elo  Champion %
1                    Spain          2  2014.1       7.633
2                Argentina          3  1967.1       6.885
3                   France          1  1915.8       6.798
4                  England          4  1890.6       6.600
5                 Portugal          5  1867.2       5.599
6                  Morocco          8  1911.5       5.384
7                   Brazil          6  1857.0       5.327
8              Netherlands          7  1836.7       4.825
9                  Germany         10  1852.9       4.291
10                  Mexico         15  1866.3       3.538
11                 Croatia         11  1802.9       3.399
12                   Japan         18  1885.6       3.040
13                Colombia         13  1849.6       2.905
14                 Belgium          9  1750.1       2.790
15                 Uruguay         17  1791.3       2.631
16                 Senegal        